In [3]:
import numpy as np

In [4]:
# constants
S = 1361.0        #W/m^2
alpha = 0.30
sigma = 5.67e-8
C = 2.0e8         #J/m^2 * K
SECONDS_PER_YEAR = 3.154e7

In [5]:
def Q_net(T, eps):
    Q_in = S * (1.0 - alpha) / 4.0
    Q_out = eps * sigma * T**4
    return Q_in - Q_out

def dTdt(T, eps):
    """Temperature derivative dT/dt."""
    return Q_net(T, eps) / C

def bisection(eps, tol=1e-5):
    """Find equilibrium temperature using the Bisection method."""
    T_low = 200.0
    T_high = 350.0

    while (T_high - T_low) > tol:
        T_mid = (T_low + T_high) / 2.0

        if Q_net(T_mid, eps) > 0:
            T_low = T_mid
        else:
            T_high = T_mid

    return (T_low + T_high) / 2.0

def simulate(eps, T0=250.0, years=500, dt_years=5):
    """Simulating temperature evolution using Euler and RK4 methods."""
    dt = dt_years * SECONDS_PER_YEAR
    n_steps = int(years / dt_years)

    # Initialize numpy arrays for time and temperatures
    time_years = np.linspace(0, years, n_steps + 1)
    T_euler = np.zeros(n_steps + 1)
    T_rk4 = np.zeros(n_steps + 1)

    T_euler[0] = T0
    T_rk4[0] = T0

    for i in range(n_steps):
        # Forward Euler Step
        T_euler[i+1] = T_euler[i] + dt * dTdt(T_euler[i], eps)

        # Runge-Kutta 4 (RK4) Step
        k1 = dTdt(T_rk4[i], eps)
        k2 = dTdt(T_rk4[i] + k1 * dt / 2.0, eps)
        k3 = dTdt(T_rk4[i] + k2 * dt / 2.0, eps)
        k4 = dTdt(T_rk4[i] + k3 * dt, eps)

        T_rk4[i+1] = T_rk4[i] + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

    return time_years, T_euler, T_rk4

# 1. Bisection calculations
T_eq_no_atm = bisection(eps=1.0)
T_eq_atm = bisection(eps=0.61)

print(f"Bisection Equilibrium (eps=1.0) : {T_eq_no_atm:.2f} K")
print(f"Bisection Equilibrium (eps=0.61): {T_eq_atm:.2f} K\n")

# 2. Dynamic Simulation
t, T_eu, T_rk = simulate(eps=0.61, T0=250.0, years=500, dt_years=5)

print("Simulation Results (eps=0.61) after 500 years:")
print(f"Euler Final Temperature : {T_eu[-1]:.4f} K")
print(f"RK4 Final Temperature   : {T_rk[-1]:.4f} K")

# Calculate the maximum absolute difference between the two methods over the simulation
max_diff = np.max(np.abs(T_eu - T_rk))
print(f"Max divergence between Euler and RK4: {max_diff:.6e} K")

Bisection Equilibrium (eps=1.0) : 254.58 K
Bisection Equilibrium (eps=0.61): 288.07 K

Simulation Results (eps=0.61) after 500 years:
Euler Final Temperature : 148.0002 K
RK4 Final Temperature   : 288.0685 K
Max divergence between Euler and RK4: 1.400685e+02 K
